# Decoder Model Sanity Check for NER

Run this **before** any NER training to verify:
1. Model loads correctly from checkpoint
2. Hidden states are valid (no NaN/Inf, reasonable norms)
3. NER head produces sensible gradients
4. The model can differentiate entity vs non-entity tokens
5. Forward pass speed is acceptable

**CPU or GPU — no training happens here.**

In [ ]:
# Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, sys, json, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA L4


In [ ]:
# ── Config (must match Phase 1 exactly) ──
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int      = 40_000
    d_model: int         = 768
    n_layers: int        = 19
    n_query_heads: int   = 12
    n_kv_heads: int      = 4
    d_ff: int            = 2048
    max_seq_len: int     = 512
    dropout: float       = 0.0    # no dropout for inference
    weight_tying: bool   = True
    rope_theta: float    = 10000.0
    def __post_init__(self):
        self.head_dim = self.d_model // self.n_query_heads
        self.n_groups = self.n_query_heads // self.n_kv_heads

CFG = Config()

In [ ]:
# ── Model architecture (same as Phase 1) ──
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], dim=-1)
    return x * c_exp + rotated * s_exp

class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh = cfg.n_query_heads
        self.nkh = cfg.n_kv_heads
        self.ng  = cfg.n_groups
        self.hd  = cfg.head_dim
        self.dm  = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1,2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        q = apply_rope(q, cos, sin, offset)
        k = apply_rope(k, cos, sin, offset)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd)
            k = k.reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd)
            v = v.reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = GQA(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, cache = self.attn(self.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x, cache

class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop     = nn.Dropout(cfg.dropout)
        self.layers   = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm     = RMSNorm(cfg.d_model)
        self.lm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.weight_tying:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len * 2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _causal_mask(self, T, device, S=None):
        if S is None: S = T
        m = torch.full((T, S), float("-inf"), device=device)
        off = S - T
        for i in range(T):
            m[i, :off+i+1] = 0.0
        return m.unsqueeze(0).unsqueeze(0)

    def forward(self, ids, targets=None, kv_caches=None, offset=0):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device, T + offset)
        new_caches = []
        for i, layer in enumerate(self.layers):
            c = kv_caches[i] if kv_caches else None
            x, nc = layer(x, self.rope_cos, self.rope_sin, mask, c, offset)
            new_caches.append(nc)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size),
                                   targets.view(-1), ignore_index=-1)
        return logits, loss, new_caches

    def get_hidden_states(self, ids):
        """Extract hidden states from the last layer (what NER uses)."""
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device)
        for layer in self.layers:
            x, _ = layer(x, self.rope_cos, self.rope_sin, mask)
        return self.norm(x)  # (B, T, d_model)

In [ ]:
# ── Load tokenizer ──
import shutil, glob

DRIVE_ROOT = "/content/drive/MyDrive"

def find_file(filename):
    """Recursively search Google Drive for a file."""
    if not os.path.isdir(DRIVE_ROOT):
        raise FileNotFoundError(f"Google Drive not mounted! Run drive.mount() first.")
    matches = glob.glob(os.path.join(DRIVE_ROOT, "**", filename), recursive=True)
    if matches:
        print(f"  Found {filename}: {matches[0]}")
        return matches[0]
    raise FileNotFoundError(f"{filename} not found anywhere in Google Drive!")

# Find and copy tokenizer files to /content so imports work
tok_json_path = find_file("bpe_40k.json")
tok_py_path = find_file("a1_tokenizer.py")

shutil.copy(tok_json_path, "/content/bpe_40k.json")
shutil.copy(tok_py_path, "/content/a1_tokenizer.py")
sys.path.insert(0, "/content")

from a1_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load("/content/bpe_40k.json")
print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")

  Found bpe_40k.json: /content/drive/MyDrive/bpe_40k.json
  Found a1_tokenizer.py: /content/drive/MyDrive/a1_tokenizer.py
Tokenizer loaded: vocab_size=40000


In [ ]:
# ── Load checkpoint ──
def find_checkpoint():
    """Search Google Drive for best checkpoint: best.pt first, then highest step_*.pt."""
    search_roots = [DRIVE_ROOT]
    # First: look for best.pt anywhere
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for match in glob.glob(os.path.join(root, "**", "best.pt"), recursive=True):
            print(f"  Found best.pt: {match}")
            return match

    # Second: find highest step_*.pt
    all_steps = []
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for match in glob.glob(os.path.join(root, "**", "step_*.pt"), recursive=True):
            # Extract step number
            base = os.path.basename(match)
            try:
                step_num = int(base.replace("step_", "").replace(".pt", ""))
                all_steps.append((step_num, match))
            except ValueError:
                continue
    if all_steps:
        all_steps.sort(reverse=True)
        best_step, best_path = all_steps[0]
        print(f"  Found highest checkpoint: step_{best_step} at {best_path}")
        return best_path

    raise FileNotFoundError(
        "No checkpoint found! Mount your Google Drive (Colab) or upload checkpoint.\n"
        "  Colab: from google.colab import drive; drive.mount('/content/drive')\n"
        "  Then place your .pt file anywhere in your Drive."
    )

CKPT_PATH = find_checkpoint()
print(f"  Using checkpoint: {CKPT_PATH}")

ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model = LegalLM(CFG).to(DEVICE)

# Load weights (handle both formats: full checkpoint dict or bare state_dict)
if "model" in ckpt:
    model.load_state_dict(ckpt["model"])
    step = ckpt.get("step", "?")
    val_loss = ckpt.get("val_loss", "?")
    print(f"Loaded checkpoint: step={step}, val_loss={val_loss}")
else:
    model.load_state_dict(ckpt)
    print("Loaded bare state_dict")

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,} ({n_params/1e6:.1f}M)")

passed = 0
failed = 0
def check(name, condition, detail=""):
    global passed, failed
    if condition:
        print(f"  [PASS] {name}")
        passed += 1
    else:
        print(f"  [FAIL] {name}  -- {detail}")
        failed += 1

  Found best.pt: /content/drive/MyDrive/best.pt
  Using checkpoint: /content/drive/MyDrive/best.pt
Loaded checkpoint: step=21500, val_loss=2.3908054494857787
Model parameters: 150,287,616 (150.3M)


## Test 1: Model loads and basic sanity

In [ ]:
print("=" * 65)
print("TEST 1: Model Load & Parameter Sanity")
print("=" * 65)

check("Parameter count in range [50M, 300M]",
      50_000_000 <= n_params <= 300_000_000, f"{n_params:,}")

# Check no NaN/Inf in any parameter
nan_params = []
inf_params = []
for name, p in model.named_parameters():
    if torch.isnan(p).any():
        nan_params.append(name)
    if torch.isinf(p).any():
        inf_params.append(name)

check("No NaN in parameters", len(nan_params) == 0,
      f"NaN in: {nan_params[:5]}")
check("No Inf in parameters", len(inf_params) == 0,
      f"Inf in: {inf_params[:5]}")

# Check parameter norms are reasonable
for name, p in model.named_parameters():
    norm = p.float().norm().item()
    if "norm" not in name and "rope" not in name:
        check(f"Param norm '{name[:40]}' < 1000",
              norm < 1000, f"norm={norm:.1f}")
        break  # just check a few

# Weight tying check
check("Weight tying active (tok_emb.weight is lm_head.weight)",
      model.tok_emb.weight.data_ptr() == model.lm_head.weight.data_ptr())

TEST 1: Model Load & Parameter Sanity
  [PASS] Parameter count in range [50M, 300M]
  [PASS] No NaN in parameters
  [PASS] No Inf in parameters
  [PASS] Param norm 'tok_emb.weight' < 1000
  [PASS] Weight tying active (tok_emb.weight is lm_head.weight)


## Test 2: Forward pass produces valid output

In [ ]:
print("\n" + "=" * 65)
print("TEST 2: Forward Pass Validity")
print("=" * 65)

test_texts = [
    "The Supreme Court held that the appellant was entitled to relief under Section 302 of the Indian Penal Code.",
    "Justice R.K. Sharma of the Delhi High Court delivered the judgment on 15th March 2021.",
]

with torch.no_grad():
    for text in test_texts:
        ids = tokenizer.encode(text)
        inp = torch.tensor([ids], dtype=torch.long, device=DEVICE)

        # Logits check
        logits, _, _ = model(inp)
        check(f"Logits shape correct for '{text[:30]}...'",
              logits.shape == (1, len(ids), CFG.vocab_size),
              f"got {logits.shape}")
        check("Logits no NaN", not torch.isnan(logits).any())
        check("Logits no Inf", not torch.isinf(logits).any())

        # Hidden states check
        hidden = model.get_hidden_states(inp)
        check(f"Hidden shape = (1, {len(ids)}, {CFG.d_model})",
              hidden.shape == (1, len(ids), CFG.d_model),
              f"got {hidden.shape}")
        check("Hidden no NaN", not torch.isnan(hidden).any())
        check("Hidden no Inf", not torch.isinf(hidden).any())

        # Hidden state norms should be reasonable (not collapsed or exploded)
        norms = hidden.float().norm(dim=-1)
        mean_norm = norms.mean().item()
        check(f"Mean hidden norm reasonable (1-100): {mean_norm:.2f}",
              1.0 < mean_norm < 100.0, f"norm={mean_norm:.2f}")


TEST 2: Forward Pass Validity
  [PASS] Logits shape correct for 'The Supreme Court held that th...'
  [PASS] Logits no NaN
  [PASS] Logits no Inf
  [PASS] Hidden shape = (1, 19, 768)
  [PASS] Hidden no NaN
  [PASS] Hidden no Inf
  [PASS] Mean hidden norm reasonable (1-100): 90.33
  [PASS] Logits shape correct for 'Justice R.K. Sharma of the Del...'
  [PASS] Logits no NaN
  [PASS] Logits no Inf
  [PASS] Hidden shape = (1, 15, 768)
  [PASS] Hidden no NaN
  [PASS] Hidden no Inf
  [PASS] Mean hidden norm reasonable (1-100): 89.67


## Test 3: Hidden states are NOT collapsed (tokens have different representations)

In [ ]:
print("\n" + "=" * 65)
print("TEST 3: Hidden State Diversity (NOT collapsed)")
print("=" * 65)

text = "Justice R.K. Sharma of the Delhi High Court delivered the judgment on 15th March 2021 in Case No. 1234."
ids = tokenizer.encode(text)
inp = torch.tensor([ids], dtype=torch.long, device=DEVICE)

with torch.no_grad():
    hidden = model.get_hidden_states(inp)[0]  # (T, d_model)

# Cosine similarity between consecutive tokens
cos_sims = []
for i in range(len(ids) - 1):
    sim = F.cosine_similarity(hidden[i].unsqueeze(0), hidden[i+1].unsqueeze(0)).item()
    cos_sims.append(sim)

mean_sim = np.mean(cos_sims)
check(f"Mean cosine sim between consecutive tokens < 0.99: {mean_sim:.4f}",
      mean_sim < 0.99,
      "Hidden states are nearly identical — model may be collapsed!")

# Check variance across token dimension
token_var = hidden.float().var(dim=0).mean().item()
check(f"Variance across tokens > 0.001: {token_var:.4f}",
      token_var > 0.001,
      "Very low variance — hidden states may be collapsed")

# Check that different tokens get different hidden states
# Pick two very different tokens
idx_justice = 0  # first token (from "Justice")
idx_2021 = -2    # near-end token (from "2021" area)
sim_far = F.cosine_similarity(
    hidden[idx_justice].unsqueeze(0),
    hidden[idx_2021].unsqueeze(0)
).item()
check(f"Distant tokens have different hidden states (sim={sim_far:.4f})",
      sim_far < 0.95)

print(f"\n  Cosine sim stats: min={min(cos_sims):.4f}, max={max(cos_sims):.4f}, mean={mean_sim:.4f}")


TEST 3: Hidden State Diversity (NOT collapsed)
  [PASS] Mean cosine sim between consecutive tokens < 0.99: 0.2825
  [PASS] Variance across tokens > 0.001: 9.1493
  [PASS] Distant tokens have different hidden states (sim=0.0715)

  Cosine sim stats: min=0.0647, max=0.4932, mean=0.2825


## Test 4: LM Head produces valid probabilities (model learned something)

In [ ]:
print("\n" + "=" * 65)
print("TEST 4: Language Model Competence")
print("=" * 65)

prompts_and_continuations = [
    ("The Supreme Court", "held"),      # common legal continuation
    ("Section 302 of the Indian", "Penal"),
    ("The learned counsel for the", "respondent"),
]

with torch.no_grad():
    for prompt, expected_word in prompts_and_continuations:
        ids = tokenizer.encode(prompt)
        inp = torch.tensor([ids], dtype=torch.long, device=DEVICE)
        logits, _, _ = model(inp)

        # Get top-5 predictions for next token
        next_logits = logits[0, -1, :]
        probs = F.softmax(next_logits, dim=-1)
        top5_ids = torch.topk(probs, 5).indices.tolist()
        top5_tokens = [tokenizer.decode([tid]) for tid in top5_ids]
        top5_probs = [probs[tid].item() for tid in top5_ids]

        # Check if expected word is in top-20
        top20_ids = torch.topk(probs, 20).indices.tolist()
        top20_tokens = [tokenizer.decode([tid]).strip().lower() for tid in top20_ids]
        in_top20 = expected_word.lower() in " ".join(top20_tokens)

        print(f"\n  Prompt: '{prompt}'")
        print(f"  Expected next: '{expected_word}'")
        print(f"  Top-5 predictions: {list(zip(top5_tokens, [f'{p:.3f}' for p in top5_probs]))}")
        check(f"'{expected_word}' plausible (in top-20 or reasonable alternative)",
              in_top20 or top5_probs[0] > 0.01,
              f"top-5: {top5_tokens}")

        # Entropy check — trained model should have lower entropy than random
        entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
        max_entropy = math.log(CFG.vocab_size)  # uniform distribution
        check(f"Entropy well below random ({entropy:.1f} << {max_entropy:.1f})",
              entropy < max_entropy * 0.5)


TEST 4: Language Model Competence

  Prompt: 'The Supreme Court'
  Expected next: 'held'
  Top-5 predictions: [('in', '0.360'), ('has', '0.156'), ('held', '0.081'), ('of', '0.061'), ('had', '0.023')]
  [PASS] 'held' plausible (in top-20 or reasonable alternative)
  [PASS] Entropy well below random (3.2 << 10.6)

  Prompt: 'Section 302 of the Indian'
  Expected next: 'Penal'
  Top-5 predictions: [('Penal', '0.999'), ('penal', '0.000'), ('Evidence', '0.000'), ('Indian', '0.000'), ('I.P.C.', '0.000')]
  [PASS] 'Penal' plausible (in top-20 or reasonable alternative)
  [PASS] Entropy well below random (0.0 << 10.6)

  Prompt: 'The learned counsel for the'
  Expected next: 'respondent'
  Top-5 predictions: [('petitioner', '0.336'), ('petitioners', '0.126'), ('appellant', '0.093'), ('applicant', '0.086'), ('respondents', '0.054')]
  [PASS] 'respondent' plausible (in top-20 or reasonable alternative)
  [PASS] Entropy well below random (2.9 << 10.6)


## Test 5: NER head gradient flow

In [ ]:
print("\n" + "=" * 65)
print("TEST 5: NER Head Gradient Flow")
print("=" * 65)

NUM_NER_TAGS = 25  # 12 labels * 2 (B/I) + O

# Simulate what happens during NER training
ner_head = nn.Linear(CFG.d_model, NUM_NER_TAGS).to(DEVICE)

text = "Justice R.K. Sharma of the Delhi High Court delivered the judgment."
ids = tokenizer.encode(text)
inp = torch.tensor([ids], dtype=torch.long, device=DEVICE)

# Enable gradients for this test
model.train()
for p in model.parameters():
    p.requires_grad_(True)

hidden = model.get_hidden_states(inp)  # (1, T, 768)
ner_logits = ner_head(hidden)          # (1, T, 25)

check(f"NER logits shape = (1, {len(ids)}, {NUM_NER_TAGS})",
      ner_logits.shape == (1, len(ids), NUM_NER_TAGS))

# Fake BIO tags (all O=0 except simulate a JUDGE entity on first few tokens)
fake_tags = torch.zeros(1, len(ids), dtype=torch.long, device=DEVICE)
fake_tags[0, 0] = 1   # B-APP (just to test gradient flow)
fake_tags[0, 1] = 2   # I-APP

loss = F.cross_entropy(ner_logits.view(-1, NUM_NER_TAGS), fake_tags.view(-1))
loss.backward()

check(f"NER loss is finite: {loss.item():.4f}", math.isfinite(loss.item()))

# Check gradients flow through to model backbone
backbone_grads = []
for name, p in model.named_parameters():
    if p.grad is not None:
        grad_norm = p.grad.float().norm().item()
        backbone_grads.append((name, grad_norm))

check("Gradients flow to backbone (at least 10 params have grad)",
      len(backbone_grads) >= 10,
      f"only {len(backbone_grads)} params got gradients")

# Check gradient norms are not zero or exploding
if backbone_grads:
    norms = [g for _, g in backbone_grads]
    check(f"Gradient norms non-zero (mean={np.mean(norms):.6f})",
          np.mean(norms) > 1e-10)
    check(f"Gradient norms not exploding (max={max(norms):.4f})",
          max(norms) < 1000)
    print(f"\n  Sample gradient norms:")
    for name, norm in backbone_grads[:5]:
        print(f"    {name[:50]:50s} grad_norm={norm:.6f}")

# Cleanup
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
    p.grad = None


TEST 5: NER Head Gradient Flow
  [PASS] NER logits shape = (1, 11, 25)
  [PASS] NER loss is finite: 6.6058
  [PASS] Gradients flow to backbone (at least 10 params have grad)
  [PASS] Gradient norms non-zero (mean=2.632505)
  [PASS] Gradient norms not exploding (max=15.8603)

  Sample gradient norms:
    tok_emb.weight                                     grad_norm=15.860306
    layers.0.norm1.weight                              grad_norm=0.348616
    layers.0.attn.q.weight                             grad_norm=1.075551
    layers.0.attn.k.weight                             grad_norm=1.075971
    layers.0.attn.v.weight                             grad_norm=11.056572


## Test 6: Speed benchmark

In [ ]:
print("\n" + "=" * 65)
print("TEST 6: Forward Pass Speed")
print("=" * 65)

# Benchmark: how fast can we process batches?
batch_sizes = [1, 4, 8]
seq_len = 510  # typical NER window

model.eval()
with torch.no_grad():
    for bs in batch_sizes:
        inp = torch.randint(0, CFG.vocab_size, (bs, seq_len), device=DEVICE)

        # Warmup
        for _ in range(3):
            _ = model.get_hidden_states(inp)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.time()
        n_iters = 20
        for _ in range(n_iters):
            _ = model.get_hidden_states(inp)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - t0
        samples_per_sec = (n_iters * bs) / elapsed
        ms_per_sample = 1000 * elapsed / (n_iters * bs)

        print(f"  batch_size={bs:2d}, seq_len={seq_len}: "
              f"{samples_per_sec:.1f} samples/sec ({ms_per_sample:.1f} ms/sample)")
        check(f"Speed OK at bs={bs} (>{0.5} samples/sec)",
              samples_per_sec > 0.5)


TEST 6: Forward Pass Speed
  batch_size= 1, seq_len=510: 31.9 samples/sec (31.4 ms/sample)
  [PASS] Speed OK at bs=1 (>0.5 samples/sec)
  batch_size= 4, seq_len=510: 39.7 samples/sec (25.2 ms/sample)
  [PASS] Speed OK at bs=4 (>0.5 samples/sec)
  batch_size= 8, seq_len=510: 39.4 samples/sec (25.4 ms/sample)
  [PASS] Speed OK at bs=8 (>0.5 samples/sec)


## Test 7: Memory usage estimate

In [ ]:
print("\n" + "=" * 65)
print("TEST 7: GPU Memory Usage")
print("=" * 65)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Simulate NER training forward+backward
    model.train()
    for p in model.parameters():
        p.requires_grad_(True)

    ner_head_test = nn.Linear(CFG.d_model, NUM_NER_TAGS).to(DEVICE)
    inp = torch.randint(0, CFG.vocab_size, (8, 510), device=DEVICE)
    tags = torch.zeros(8, 510, dtype=torch.long, device=DEVICE)

    hidden = model.get_hidden_states(inp)
    logits = ner_head_test(hidden)
    loss = F.cross_entropy(logits.view(-1, NUM_NER_TAGS), tags.view(-1))
    loss.backward()

    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Peak GPU memory (bs=8, seq=510, full fine-tune): {peak_mem:.2f} GB")

    if peak_mem < 16:
        print(f"  Fits on T4 (16 GB)")
    elif peak_mem < 40:
        print(f"  Needs A100-40GB or larger")
    else:
        print(f"  Needs A100-80GB — consider LoRA or gradient checkpointing")

    check(f"Memory usage under 80GB", peak_mem < 80)

    # Cleanup
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
        p.grad = None
    del ner_head_test, hidden, logits, loss, inp, tags
    torch.cuda.empty_cache()
else:
    print("  (Skipping GPU memory test — running on CPU)")


TEST 7: GPU Memory Usage
  Peak GPU memory (bs=8, seq=510, full fine-tune): 9.46 GB
  Fits on T4 (16 GB)
  [PASS] Memory usage under 80GB


## Test 8: Different tokens produce different hidden states (critical for NER)

In [ ]:
print("\n" + "=" * 65)
print("TEST 8: Entity-Relevant Representations")
print("=" * 65)

# The key question: does the model produce different representations
# for tokens that are entities vs tokens that are not?
text = "Justice Sharma delivered the judgment in Delhi High Court on 15 March 2021 under Section 302 IPC"
ids = tokenizer.encode(text)
inp = torch.tensor([ids], dtype=torch.long, device=DEVICE)

with torch.no_grad():
    hidden = model.get_hidden_states(inp)[0]  # (T, 768)

# Print what each token maps to
print(f"\n  Token representations for: '{text}'")
print(f"  {'Idx':>4} {'Token':<20} {'Norm':>8} {'Mean':>8} {'Std':>8}")
print("  " + "-" * 55)
for i, tid in enumerate(ids):
    tok_text = tokenizer.decode([tid])
    h = hidden[i].float()
    print(f"  {i:4d} {tok_text:<20} {h.norm():.4f} {h.mean():.4f} {h.std():.4f}")

# Check that norms vary (not all the same)
norms = [hidden[i].float().norm().item() for i in range(len(ids))]
norm_std = np.std(norms)
check(f"Hidden norm variance across tokens > 0 (std={norm_std:.4f})",
      norm_std > 0.01, "All tokens have identical norms — model is dead")


TEST 8: Entity-Relevant Representations

  Token representations for: 'Justice Sharma delivered the judgment in Delhi High Court on 15 March 2021 under Section 302 IPC'
   Idx Token                    Norm     Mean      Std
  -------------------------------------------------------
     0 Justice              91.9676 -0.1572 3.3170
     1 Sharma               88.6540 -0.1477 3.1977
     2 delivered            88.9690 0.0024 3.2125
     3 the                  90.3114 0.2805 3.2489
     4 judgment             89.7799 0.1486 3.2384
     5 in                   90.1389 -0.0387 3.2545
     6 Delhi                89.3955 -0.3659 3.2070
     7 High                 88.8383 0.1337 3.2050
     8 Court                89.8353 0.0137 3.2437
     9 on                   89.1674 -0.0377 3.2194
    10 15                   94.1289 -0.1387 3.3960
    11 March                90.9475 -0.0366 3.2837
    12 2021                 90.0544 0.1538 3.2480
    13 under                89.2522 0.2474 3.2132
    14 Sec

## Summary

In [ ]:
print("\n" + "=" * 65)
print(f"RESULTS: {passed} passed, {failed} failed out of {passed + failed} tests")
print("=" * 65)

if failed == 0:
    print("\nALL TESTS PASSED!")
    print("Your decoder model is ready to serve as NER backbone.")
    print("\nKey numbers to remember:")
    print(f"  - Model: {n_params/1e6:.1f}M params, d_model={CFG.d_model}")
    print(f"  - Hidden states: shape (batch, seq_len, {CFG.d_model})")
    print(f"  - NER head needed: nn.Linear({CFG.d_model}, {NUM_NER_TAGS})")
    print(f"  - Gradients flow correctly from NER loss through full backbone")
else:
    print(f"\n!! {failed} FAILURES — fix these before NER training!")


RESULTS: 38 passed, 0 failed out of 38 tests

ALL TESTS PASSED!
Your decoder model is ready to serve as NER backbone.

Key numbers to remember:
  - Model: 150.3M params, d_model=768
  - Hidden states: shape (batch, seq_len, 768)
  - NER head needed: nn.Linear(768, 25)
  - Gradients flow correctly from NER loss through full backbone
